# CNN 1D Multivariate

In this section we implement multivariate forecasting using the CNN 1D Model (1D Convolutional Neural Network) with the TimeSeriesDatasetVectorizedExog approach.

The CNN 1D (1D Convolutional Neural Network) Forecaster is the same univariate model used in the univariate approach, but extended to multivariate forecasting through efficient batching. Instead of processing series individually, TimeSeriesDatasetVectorizedExog batches all 1502 series together, allowing the univariate model to train on multiple series simultaneously with exogenous features (GDP, CPI, Interest Rate).

With this approach the model architecture remains unchanged - we simply reshape the data to process all series in parallel, achieving faster training while incorporating exogenous variables. Convolutional filters capture local temporal patterns across all series efficiently.

## Architecture

```bash
Input (seq_length, input_size)
    ↓
Transpose (input_size, seq_length)
    ↓
Conv1D Block 1
  ├─ Conv1D (input_size → hidden_size, kernel=3)
  ├─ ReLU
  └─ MaxPool1D
    ↓
Conv1D Block 2
  ├─ Conv1D (hidden_size → hidden_size, kernel=3)
  ├─ ReLU
  └─ MaxPool1D
    ↓
Conv1D Block 3
  └─ (same structure)
    ↓
Adaptive Average Pooling (→ 1)
    ↓
Flatten
    ↓
Fully Connected (hidden_size → hidden_size)
    ↓
ReLU + Dropout
    ↓
Fully Connected (hidden_size → 1)
    ↓
Output (1 prediction)
```

## Model

In [ ]:
import torch 
import torch.nn as nn

In [ ]:
class CNN1DForecaster(nn.Module):
    """
    1D CNN model for MULTIVARIATE time series forecasting.
    Architecture: 
        Conv1D blocks (Conv -> ReLU -> MaxPool) -> 
        Adaptive Average Pooling -> Flatten -> 
        Fully Connected -> Dropout -> Output
    
    CNNs can capture local patterns and temporal dependencies efficiently.
    Uses multiple kernel sizes to capture patterns at different scales.
    Processes sequences in parallel (unlike RNN/LSTM/GRU).
    """
    def __init__(self, input_size, hidden_size=64, num_layers=3, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            hidden_size: Number of filters in conv layers
            num_layers: Number of convolutional blocks (minimum 1)
            dropout: Dropout rate
        """
        super(CNN1DForecaster, self).__init__()
        
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = max(1, num_layers)
        
        # Convolutional layers
        conv_layers = []
        
        # First conv block: input_size -> hidden_size
        conv_layers.extend([
            nn.Conv1d(in_channels=input_size, out_channels=hidden_size, 
                     kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=1, padding=1)
        ])
        
        # Additional conv blocks: hidden_size -> hidden_size
        for i in range(1, self.num_layers):
            conv_layers.extend([
                nn.Conv1d(in_channels=hidden_size, out_channels=hidden_size, 
                         kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool1d(kernel_size=2, stride=1, padding=1)
            ])
        
        self.conv_blocks = nn.Sequential(*conv_layers)
        
        # Adaptive pooling to fixed size output
        self.adaptive_pool = nn.AdaptiveAvgPool1d(1)
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # Conv1d expects (batch_size, channels, seq_length)
        # Transpose from (batch, seq, features) to (batch, features, seq)
        x = x.transpose(1, 2)  # (batch_size, input_size, seq_length)
        
        # Apply convolutional blocks
        x = self.conv_blocks(x)  # (batch_size, hidden_size, seq_length')
        
        # Adaptive pooling to reduce to (batch_size, hidden_size, 1)
        x = self.adaptive_pool(x)  # (batch_size, hidden_size, 1)
        
        # Flatten
        x = x.squeeze(-1)  # (batch_size, hidden_size)
        
        # Fully connected layers
        x = self.fc1(x)  # (batch_size, hidden_size)
        x = self.relu(x)
        x = self.dropout(x)
        out = self.fc2(x)  # (batch_size, 1)
        
        return out


## Model Results without Exogenous Features

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration (seconds) |
|-------|-----------------|------------|---------|-------------|---------------|------------|-------------------|
| 0 | 0.12568 | 4 | 0.1656 | 32 | 0.006161 | 2 | 2.47 |
| 1 | 0.12606 | 4 | 0.1637 | 128 | 0.001886 | 4 | 10.38 |
| 2 | 0.15506 | 16 | 0.3782 | 32 | 0.000342 | 4 | 3.38 |
| 3 | 0.12880 | 16 | 0.2340 | 64 | 0.007972 | 2 | 4.64 |
| 4 | 0.13128 | 4 | 0.1508 | 64 | 0.000422 | 3 | 6.11 |


### Best Hyperparameters 

Parameters for Trial 0:
- learning_rate: 0.0061
- batch_size: 4
- num_layers: 2
- hidden_size: 32
- dropout: 0.1656



#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/multivariate/cnn1d/fold1/fold_results.png)


#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/multivariate/cnn1d/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

    
![Fold 3 Results](./img/multivariate/cnn1d/fold3/fold_results.png)


### Fold Results

| Fold | MSE | RMSE | MAE | R² | SMAPE (%) |
|------|---------|---------|---------|--------|-----------|
| Fold 1 | 63524.60 | 252.04 | 110.37 | 0.8741 | 62.13 |
| Fold 2 | 53075.31 | 230.38 | 101.68 | 0.8828 | 85.34 |
| Fold 3 | 49273.36 | 221.98 | 92.98 | 0.8987 | 80.40 |
| **Average** | **55291.09 ± 7379.48** | **234.80 ± 15.51** | **101.68 ± 8.70** | **0.8852 ± 0.0125** | **75.96 ± 12.23** |

### SMAPE Distribution Accross Folds

| SMAPE Range | Series Percentage | Number of Series (avg) |
|-------------|-------------------|------------------------|
| <10% | 7.8% ± 8.8% | 117 |
| 10-20% | 11.1% ± 1.2% | 167 |
| 20-30% | 13.0% ± 3.0% | 196 |
| 30-40% | 10.8% ± 1.7% | 162 |
| >40% | 57.2% ± 4.6% | 859 |

### Comparison to Baseline Model

The CNN 1D multivariate model achieves an average SMAPE of 75.96% ± 12.23%, which is **3.70 percentage points higher** than the baseline 3-month rolling average (72.26% ± 7.06%). While the CNN 1D model shows competitive performance, it exhibits higher variation across folds (standard deviation: 12.23% vs 7.06%) compared to the baseline. The model performs best in Fold 1 (62.13% SMAPE), demonstrating its ability to capture local temporal patterns through convolutional filters. However, the overall underperformance relative to the baseline suggests that the CNN 1D architecture, despite its efficiency in parallel processing, may not fully capture the complex long-term dependencies present in this multivariate time series data as effectively as the simpler rolling average approach.


## Model Results with Exogenous Features

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration (seconds) |
|-------|-----------------|------------|---------|-------------|---------------|------------|-------------------|
| 0 | 0.22857 | 16 | 0.1186 | 128 | 0.001502 | 4 | 8.80 |
| 1 | 0.44867 | 16 | 0.2984 | 32 | 0.002470 | 4 | 1.31 |
| 2 | 0.20250 | 8 | 0.1682 | 64 | 0.000615 | 3 | 4.26 |
| 3 | 0.32231 | 4 | 0.1278 | 64 | 0.007431 | 2 | 1.67 |
| 4 | 0.18494 | 4 | 0.1391 | 128 | 0.002452 | 4 | 13.03 |

### Best Hyperparameters

Parameters for Trial 4:
- learning_rate: 0.00245
- batch_size: 4
- num_layers: 4
- hidden_size: 128
- dropout: 0.13905


#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/multivariate/cnn1d_exog/fold1/fold_results.png)


#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/multivariate/cnn1d_exog/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

    
![Fold 3 Results](./img/multivariate/cnn1d_exog/fold3/fold_results.png)



### Fold Results

| Fold | MSE | RMSE | MAE | R² | SMAPE (%) |
|------|---------|---------|---------|--------|-----------|
| Fold 1 | 143773.92 | 379.18 | 320.05 | 0.7151 | 128.33 |
| Fold 2 | 74069.35 | 272.16 | 121.63 | 0.8364 | 96.06 |
| Fold 3 | 100965.59 | 317.75 | 265.03 | 0.7924 | 125.83 |
| **Average** | **106269.62 ± 35153.68** | **323.03 ± 53.70** | **235.57 ± 102.44** | **0.7813 ± 0.0614** | **116.74 ± 17.95** |

### SMAPE Distribution Accross Folds

| SMAPE Range | Series Percentage | Number of Series (avg) |
|-------------|-------------------|------------------------|
| <10% | 1.8% ± 0.4% | 27 |
| 10-20% | 6.6% ± 1.0% | 99 |
| 20-30% | 8.3% ± 2.2% | 124 |
| 30-40% | 6.7% ± 1.6% | 100 |
| >40% | 76.7% ± 4.5% | 1152 |

### Comparison to Baseline Model

The CNN 1D multivariate model with exogenous features achieves an average SMAPE of 116.74% ± 17.95%, which is **44.48 percentage points higher** than the baseline 3-month rolling average (72.26% ± 7.06%). The inclusion of exogenous features (GDP, CPI, Interest Rate) significantly degrades performance compared to both the baseline and the CNN 1D model without exogenous features (75.96% ± 12.23%). The model exhibits substantially higher variation across folds (standard deviation: 17.95% vs 7.06%) and performs worst in Fold 1 (128.33% SMAPE). Notably, 76.7% of series have SMAPE >40%, indicating widespread forecasting difficulties. This substantial underperformance suggests that the CNN 1D architecture struggles to effectively integrate exogenous variables with temporal patterns, and the additional complexity introduced by these features may be causing overfitting or diluting the model's ability to capture meaningful temporal dependencies in this multivariate time series context.